<a href="https://colab.research.google.com/github/Liping-LZ/BDAO_DSDO/blob/main/Week%201/API_tutorial_withtaskanswer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Extracting API data

In class, we have discussed about the key concepts of Application Programming Interface (API). There is also an overview of API basics in GitHub ([API_Fundamentals.MD](https://github.com/Liping-LZ/BDAO_DSDO_2425/blob/main/Web_crawling_and_API/API_fundamentals.MD)). We also have demonstrated how API works and we can extract data via API with some hands-on sessions. Please review what you have done in class as well.

In this tutorial, we will will show you another example of extracting data through API by building a program to interact with a series of APIs. Specifically we will be following this workflow:

1. Creating a random/fake person using the _RandomUser_ API (check [documentation](https://randomuser.me/documentation));
2. Predicting the age of the user using the _Agify.io_ API (check [documentation](https://publicapi.dev/agify-io-api));
3. Predicting if the nationality of the user using the _Nationalize.io_ API (check [documentation](https://nationalize.io/documentation)).

As you can see it's a fairly silly task, however these APIs require no security keys to access (which makes life far easier for us) and its enough to give us some basic exposure to working with APIs.

Let's begin!

In [ ]:
import json
import requests

random_user = "https://randomuser.me/api/" # this is the base url

Everything should be self explanatory in the above, we import the relevant libraries/packages (_json_ to deal with the API's JSON outputs and _requests_ to make the API calls) and we create a variable to store the API endpoint for our fake person generator (the _RandomUser_ API). The next step will be to call the API:

In [ ]:
def name_generator(url):
  r = requests.get(url)
  return r

new_user = name_generator(random_user)
new_user.content

b'{"results":[{"gender":"female","name":{"title":"Mrs","first":"Ivy","last":"White"},"location":{"street":{"number":5482,"name":"Andersons Bay Road"},"city":"Hastings","state":"Taranaki","country":"New Zealand","postcode":45522,"coordinates":{"latitude":"50.5536","longitude":"131.7694"},"timezone":{"offset":"+4:30","description":"Kabul"}},"email":"ivy.white@example.com","login":{"uuid":"7d8a805b-6abc-4f5e-9d5a-f9d3ff3916a1","username":"purplewolf618","password":"earnhard","salt":"QSW7v4f2","md5":"96fd89c1bb890e5f0967810eabaa2d84","sha1":"2fd3bda9bb8f68502116d36a7363a225bffbf6b6","sha256":"09f4f695704703f33e464c39267535b34590fc1ff2e4bcbd4dceb63ea5faa1ee"},"dob":{"date":"1992-03-12T05:29:24.154Z","age":33},"registered":{"date":"2006-04-12T15:19:27.766Z","age":19},"phone":"(925)-146-0559","cell":"(281)-943-9653","id":{"name":"","value":null},"picture":{"large":"https://randomuser.me/api/portraits/women/95.jpg","medium":"https://randomuser.me/api/portraits/med/women/95.jpg","thumbnail":"ht

Here we create a simple function to send the API request (using the _requests_ package as above). You may note the function we use is _get_ which (unsurprisingly) translates as a _GET_ HTTP request in the RESTful framework (see previous Notebook if that is not clear).

We then use our function to create a new variable (_new\_user_). Note that we are returned, in this case, a JSON object - effectively a dictionary. The _requests_ package will guess this for us so may return an XML object or similar with other APIs. From this JSON object we specify the key "content" to access the information we want. We could also specify keys such as "encoding" or "status_code" if that's the information we require (see more [here](https://docs.python-requests.org/en/latest/index.html)).

The "content" object, however, is a Python string. It would be more useful if we could interact with this as JSON:

In [ ]:
response_dict = new_user.json()
response_dict

{'results': [{'gender': 'female',
   'name': {'title': 'Mrs', 'first': 'Ivy', 'last': 'White'},
   'location': {'street': {'number': 5482, 'name': 'Andersons Bay Road'},
    'city': 'Hastings',
    'state': 'Taranaki',
    'country': 'New Zealand',
    'postcode': 45522,
    'coordinates': {'latitude': '50.5536', 'longitude': '131.7694'},
    'timezone': {'offset': '+4:30', 'description': 'Kabul'}},
   'email': 'ivy.white@example.com',
   'login': {'uuid': '7d8a805b-6abc-4f5e-9d5a-f9d3ff3916a1',
    'username': 'purplewolf618',
    'password': 'earnhard',
    'salt': 'QSW7v4f2',
    'md5': '96fd89c1bb890e5f0967810eabaa2d84',
    'sha1': '2fd3bda9bb8f68502116d36a7363a225bffbf6b6',
    'sha256': '09f4f695704703f33e464c39267535b34590fc1ff2e4bcbd4dceb63ea5faa1ee'},
   'dob': {'date': '1992-03-12T05:29:24.154Z', 'age': 33},
   'registered': {'date': '2006-04-12T15:19:27.766Z', 'age': 19},
   'phone': '(925)-146-0559',
   'cell': '(281)-943-9653',
   'id': {'name': '', 'value': None},
   'pi

Much prettier. `json()` is a method that is used to convert the JSON response into a Python dictionary, making it easier to work with. Now we can use the JSON keys to get individual items in this output:

In [ ]:
response_dict['results'][0]['name']['first']

'Ivy'

An excellent choice of first name. Note the way we need to query the object - this is not always obvious and you need to look carefully at the object being returned. In this case the parent node is a choice of either 'results' or 'info' - we want the former. When we look at _response\_dict['results']_ we see it contains a dictionary inside a list (a list with one item). We need to specify the list index 0 to return the inner dictionary. From that dictionary within the list we can specify the key 'name' and within the nested dictionary this represents we specify the final key 'first' (i.e. firstname).

Every API returns things a bit differently. Ultimately you just need to inspect the output and adjust accordingly. we are basically getting information we want by locating the information in the dictionary (but there is list and nested dictionary with the dictionary). You need to review the concept of list and dictionary in Python if you have no ideas about how this works.

Let's tidy this up into a reusable function:


In [ ]:
def name_generator(url):
  new_user = []
  r = requests.get(url)
  response_dict = r.json()
  new_user.append({'name': response_dict['results'][0]['name']['first'], \
                 'age': response_dict['results'][0]['dob']['age'], \
                 'country': response_dict['results'][0]['location']['country']})

  return new_user

In [ ]:
new_user = name_generator(random_user)
new_user

[{'name': 'Ariadne', 'age': 47, 'country': 'Brazil'}]

Now we can generate a random user, our next task is to create a function to guess their age (using the _Agify.io_ API). This uses the following endpoint:

In [ ]:
aging_url = "https://api.agify.io/?name="

However, as you may have spotted, this is not really a full endpoint. The API needs an input (the name) to generate the ouput (the predicted age) and this input needs to be included in the endpoint.

As the input (name) will vary we will dynamically join this information to the endpoint before making the call like so:

In [ ]:
def age_generator(url, user):
  search = url + user[0]['name']
  r = requests.get(search)
  return r

Everything here is the same as before, except for us concatenating (joining) the user name to the end of the URL. I.e. if we pass the url as "https://api.agify.io/?name=" and the user we generated earlier, our joined up endpoint (the _search_ variable) would be "https://api.agify.io/?name=Ariadne". Let's test it out:

In [ ]:
x = age_generator(aging_url, new_user)
x.content

b'{"count":365,"name":"Ariadne","age":57}'

Excellent - we have our (admittedly pretty inaccurate in this case) guess! However, we also get other data we don't need so let's tidy up the function and get it to return just the predicted age:

In [ ]:
def age_generator(url, user):
  search = url + user[0]['name']
  r = requests.get(search)
  response_dict = r.json()

  return response_dict["age"]

x = age_generator(aging_url, new_user)
x

57

Again, this is all as we saw previously although you may note we have a far easier time extracting the information as there is no list or nested dictionaries (we need only pass the 'age' key).

Let's do the same for our nationality predictor (using the _Nationalize.io_ API):

In [ ]:
country_url = "https://api.nationalize.io/?name="

def country_generator(url, user):
  search = url + user[0]['name']
  r = requests.get(search)
  return r

In [ ]:
y = country_generator(country_url, new_user)
y.content

b'{"count":236,"name":"Ariadne","country":[{"country_id":"BR","probability":0.09747041791425459},{"country_id":"PA","probability":0.06402538051358805},{"country_id":"BO","probability":0.0535396738944375},{"country_id":"AO","probability":0.04448985528559026},{"country_id":"MX","probability":0.04070008483126411}]}'

Again, the Python code is basically the same as before, but we are back with the messy nested dictionaries/list. We actually get five predictions here, but fortunately they are in order of probability. To keep things simple we will just use the prediction with the highest probability (the first one):

In [ ]:
def country_generator(url, user):
  search = url + user[0]['name']
  r = requests.get(search)
  response_dict = r.json()

  return response_dict["country"][0]["country_id"]

y = country_generator(country_url, new_user)
y

'BR'

Checking online 'BR' is the country code for Brazil and it is correct! Yeah!

With these elements in place we can put it all together with a function that will create a fake user and then guess their age and nationality:

In [ ]:
def fake_user_guesser(name_url, aging_url, country_url):
  fake_user = name_generator(random_user)
  age_guess = age_generator(aging_url, fake_user)
  country_guess = country_generator(country_url, fake_user)

  return f"New user is {fake_user[0]['name']}, aged {fake_user[0]['age']} from {fake_user[0]['country']}. We guessed the age was {age_guess} and the country was {country_guess}."

In [ ]:
random_user = "https://randomuser.me/api/"
aging_url = "https://api.agify.io/?name="
country_url = "https://api.nationalize.io/?name="

output = fake_user_guesser(random_user, aging_url, country_url)
output

'New user is Magnus, aged 57 from Denmark. We guessed the age was 53 and the country was NO.'

And there we go. Our predictions are not too bad this time. 53 is close enough to 57 and Norway is close to Denmark. Well done! You might get different results because it is random user generator and we won't get the same name all the time. Give it a try yourself.

We could obviously think of many improvements to this function - such as actually getting the country name from the code generated or expanding it to compare the other nationality predictions. However, we have met our objectives!

To finish off let's make a few more calls and see how our predictions fair. However, to avoid stressing the APIs we should leave a little gap between calls. We can use the in-built _time_ to set a 5 second rest between each:

In [ ]:
import time
for i in range(10):
  output = fake_user_guesser(random_user, aging_url, country_url)
  print(output)
  if i != 9:
    print("\n")
  time.sleep(5)

New user is Allen, aged 30 from United States. We guessed the age was 76 and the country was US.


New user is Nayeli, aged 35 from Mexico. We guessed the age was 24 and the country was MX.


New user is Giuliana, aged 41 from Switzerland. We guessed the age was 63 and the country was IT.


New user is Gabriel, aged 79 from Spain. We guessed the age was 56 and the country was RO.


New user is Mario, aged 38 from Australia. We guessed the age was 63 and the country was AO.


New user is دینا, aged 67 from Iran. We guessed the age was None and the country was IR.


New user is Marcel, aged 29 from Germany. We guessed the age was 64 and the country was RO.


New user is Maria, aged 48 from Denmark. We guessed the age was 61 and the country was RO.


New user is Venla, aged 56 from Finland. We guessed the age was 32 and the country was MZ.


New user is Lotta, aged 26 from Finland. We guessed the age was 58 and the country was SE.


Overall, the function works well, though the guesses are not perfect. But you are shown how we can interact APIs to extract data and applied some built algorithms/functions to integrate information. We can easily make a software/app which integrates different information or functions by linking to different APIs. Pretty nice, right?

## Additional Tasks
While our basic functionality seems to work, there are no measures in place to deal with an API failing ... which given enough use would be a reasonable expectation. Can you rewrite the code to deal with this potential issues _Hint: the request package will allow you to check the status of the response using \<variable\>.status\_code_. Good luck!

In [ ]:
def fake_user_guesser(name_url, aging_url, country_url):
    fake_user_response = requests.get(name_url)
    if fake_user_response.status_code != 200:
        return f"Error getting user data: Status code {fake_user_response.status_code}"

    fake_user = name_generator(name_url)

    age_guess_response = requests.get(aging_url + fake_user[0]['name'])
    if age_guess_response.status_code != 200:
        return f"Error getting age guess: Status code {age_guess_response.status_code}"

    age_guess = age_generator(aging_url, fake_user)

    country_guess_response = requests.get(country_url + fake_user[0]['name'])
    if country_guess_response.status_code != 200:
        return f"Error getting country guess: Status code {country_guess_response.status_code}"

    country_guess = country_generator(country_url, fake_user)

    return f"New user is {fake_user[0]['name']}, aged {fake_user[0]['age']} from {fake_user[0]['country']}. We guessed the age was {age_guess} and the country was {country_guess}."